***

Preparing Workspace

***

In [ ]:




by_mode=False
export=False


import pandas as pd
import re
import traceback
from pathlib import Path
import plotly.express as px
pd.options.display.float_format = '{:.2f}'.format

PATH_GIT     = Path.home() / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
PATH_CONFIG0 = PATH_GIT / 'config'
PATH_CONFIG  = PATH_GIT / 'Data' / 'Census' / 'config'

PATH_PLOTS = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring")
print('Export Location: ' + str(PATH_PLOTS))


import sys
sys.path.append(str(PATH_CONFIG0))
import plot as pt


dt_mode = {
    'DR': 'Demand Response',
    'LR': 'Light Rail',
    'MB': 'Motor Bus',
    'CB': 'Commuter Bus'
}

color_map  = {
    'Trimet': '#9DC209'
    , 'RTD': '#1E90FF'
    , 'VTA': "#FBB117"
    , 'MTS': "#DC381F"
    , 'METRO': '#3B9C9C'
    , 'SacRT': "#000000"
}


def plot_NTD(export, indicator, wkbk, value, group, comp):

    file_in = PATH_PLOTS / 'Data' / f"{indicator} {wkbk}.xlsx"
    df = pd.read_excel(file_in, sheet_name='Data')

    if comp == 'peers_sacrt':
        df = df[(df['Peer Region'].isin(['sacrt_peer'])) | (df['Acronym'] == 'SacRT')]
    if comp == 'self':
        df = df[df['Peer Region'].isin(['self'])]
    
    df = df[['Year', 'Acronym', group, value]].reset_index(drop=True)
    df[value] = df[value]

    display(df.head())

    file_plot_name = re.sub(' ', '_', wkbk.lower())
    for sub in df[group].unique():

        try:
            df_plot = df.copy()
            df_plot = df_plot[df_plot[group] == sub]

            fig = px.line(df_plot, x='Year', y=value, color='Acronym', color_discrete_map=color_map, markers=True)

            if group == 'Transit Mode':
                title = f'<b>{dt_mode[sub]} {wkbk} Per Capita</b>  <br><sup>Peer Agencies</sup>'
            else:
                title = f'<b>{sub} {wkbk} per Capita</b>  <br><sup>Peer Agencies</sup>'
                
            # TODO:  Need to include edits for Transit_6 and Transit_8
            fig.update_traces(hovertemplate='%{y}')
            fig.update_yaxes(matches=None)

            plot_name = f'{re.sub(' ', '_', sub.lower())}_{file_plot_name}_{comp}'
            pt.plot_agol(fig, export, title, indicator, plot_name)

        except Exception as e: print(e); traceback.print_exc(); print(); print()




Population

In [ ]:


export=False

file_in = PATH_PLOTS / 'Data' / 'Population Estimates by Transit Agency.xlsx'
df_pop = pd.read_excel(file_in, sheet_name='Data')

indicator = 'Transit_1'
wkbk = 'Service Hours'
value = 'VRH Per Capita'

file_in = PATH_PLOTS / 'Data' / f"{indicator} {wkbk}.xlsx"
df = pd.read_excel(file_in, sheet_name='Data')

df_pop = df_pop.merge(df[['NTD ID', 'Acronym', 'Peer Region']].drop_duplicates(), on='NTD ID')
df_pop = df_pop.set_index(['Year', 'NTD ID', 'Acronym', 'Peer Region']).reset_index()
df_pop = df_pop[df_pop['Peer Region'].isin(['self'])]

display(df_pop.head())

fig = px.line(df_pop, x='Year', y='Service Area Pop', color='Acronym', markers=True)

title = f'<b>Population by Service Area</b>  <br><sup>SACOG 6-County Region</sup>'
fig.update_traces(hovertemplate='%{y}')
fig.update_yaxes(matches=None)
fig.update_layout(legend={'traceorder': 'reversed'})
fig.update_xaxes(tick0=0, dtick=1)

plot_name = f'population_by_agency'
pt.plot_agol(fig, export, title, indicator, plot_name)



Transit_1 Service

In [ ]:


indicator = 'Transit_1'
wkbk = 'Service Hours SACOG'
value = 'VRH Per Capita'


file_in = PATH_PLOTS / 'Data' / f"{indicator} {wkbk}.xlsx"
df = pd.read_excel(file_in, sheet_name='Data')

df = df[df['Transit Mode'] != 'CB']
df = df[['Year', 'Region', 'Transit Mode', value]].drop_duplicates().sort_values(['Year', value], ascending=[False,False]).reset_index(drop=True)



dt_modes = {
    'DR': 'Demand Response',
    'LR': 'Light Rail',
    'MB': 'Local Bus'
}

df['Transit Mode'] = df['Transit Mode'].map(dt_modes)
df[value] = round(df[value], 3)


display(df.head())


color_map  = {
    'Demand Response': '#9DC209'
    , 'Light Rail': '#1F45FC'
    , 'Local Bus': '#1E90FF'
}

file_plot_name = re.sub('_sacog', '', re.sub(' ', '_', wkbk.lower()))

fig = px.bar(df, x='Year', y=value, color='Transit Mode',  color_discrete_map=color_map)

wkbk = re.sub(' SACOG', '', wkbk)
title = f'<b>Transit Service Hours per Capita by Transit Mode</b>  <br><sup>SACOG 6-County Region</sup>'
    
fig.update_traces(hovertemplate='%{y}')
fig.update_yaxes(matches=None)
fig.update_layout(legend={'traceorder': 'reversed'})
fig.update_xaxes(tick0=0, dtick=1)

plot_name = f'all_modes_{file_plot_name}_sacog'
pt.plot_agol(fig, export, title, indicator, plot_name)



In [ ]:

if by_mode:
    # Set params
    indicator = 'Transit_1'
    wkbk = 'Service Hours'
    value = 'VRH Per Capita'
    group = 'Transit Mode'
    comp = 'peers_sacrt'

    # Plot
    plot_NTD(export, indicator, wkbk, value, group, comp)




In [ ]:

if by_mode:

    # Set params
    indicator = 'Transit_1'
    wkbk = 'Service Hours'
    value = 'VRH Per Capita'
    group = 'Transit Mode'
    comp = 'self'


    # Plot
    plot_NTD(export, indicator, wkbk, value, group, comp)




Transit_2 Ridership

In [ ]:


indicator = 'Transit_2'
wkbk = 'Ridership SACOG'
value = 'UPT Per Capita'


file_in = PATH_PLOTS / 'Data' / f"{indicator} {wkbk}.xlsx"
df = pd.read_excel(file_in, sheet_name='Data')

df = df[df['Transit Mode'] != 'CB']
df = df[['Year', 'Region', 'Transit Mode', value]].drop_duplicates().sort_values(['Year', value], ascending=[False,False]).reset_index(drop=True)



dt_modes = {
    'DR': 'Demand Response',
    'LR': 'Light Rail',
    'MB': 'Local Bus'
}

df['Transit Mode'] = df['Transit Mode'].map(dt_modes)
df[value] = round(df[value], 3)


display(df.head())


color_map  = {
    'Demand Response': '#9DC209'
    , 'Light Rail': '#1F45FC'
    , 'Local Bus': '#1E90FF'
}


file_plot_name = re.sub('_sacog', '', re.sub(' ', '_', wkbk.lower()))

fig = px.bar(df, x='Year', y=value, color='Transit Mode',  color_discrete_map=color_map)

title = f'<b>Annual Transit Passenger Boardings per Capita by Transit Mode</b>  <br><sup>SACOG 6-County Region</sup>'
    
fig.update_traces(hovertemplate='%{y}')
fig.update_yaxes(matches=None)
fig.update_layout(legend={'traceorder': 'reversed'})
fig.update_xaxes(tick0=0, dtick=1)

plot_name = f'all_modes_{file_plot_name}_sacog'
pt.plot_agol(fig, export, title, indicator, plot_name)



In [ ]:


if by_mode:

    # Set params
    indicator = 'Transit_2'
    wkbk = 'Ridership'
    value = 'UPT Per Capita'
    group = 'Transit Mode'
    comp = 'peers_sacrt'

    # Plot
    plot_NTD(export, indicator, wkbk, value, group, comp)




In [ ]:


if by_mode:


    # Set params
    indicator = 'Transit_2'
    wkbk = 'Ridership'
    value = 'UPT Per Capita'
    group = 'Transit Mode'
    comp = 'self'

    # Plot
    plot_NTD(export, indicator, wkbk, value, group, comp)




Transit_4 Fares

In [ ]:

if by_mode:

    # Set params
    indicator = 'Transit_4'
    wkbk = 'Fares'
    value = 'Fare Revenues Per Capita'
    group = 'Transit Mode'
    comp = 'peers_sacrt'

    # Plot
    plot_NTD(export, indicator, wkbk, value, group, comp)



In [ ]:

if by_mode:

    # Set params
    indicator = 'Transit_4'
    wkbk = 'Fares'
    value = 'Fare Revenues Per Capita'
    group = 'Transit Mode'
    comp = 'self'

    # Plot
    plot_NTD(export, indicator, wkbk, value, group, comp)



Transit_5 Operating Expenses

In [ ]:


if by_mode:

    # Set params
    indicator = 'Transit_5'
    wkbk = 'Operating Expenses'
    value = 'Total Operating Expenses Per Capita'
    group = 'Transit Mode'
    comp = 'peers_sacrt'

    # Plot
    plot_NTD(export, indicator, wkbk, value, group, comp)




In [ ]:

if by_mode:


    # Set params
    indicator = 'Transit_5'
    wkbk = 'Operating Expenses'
    value = 'Total Operating Expenses Per Capita'
    group = 'Transit Mode'
    comp = 'self'

    # Plot
    plot_NTD(export, indicator, wkbk, value, group, comp)



Transit_6 Revenue Sources

In [ ]:


if by_mode:


    # Set params
    indicator = 'Transit_6'
    wkbk = 'Revenue Sources'
    value = 'Funds Expended on Operations Per Capita'
    group = 'Source'
    comp = 'peers_sacrt'

    # Plot
    plot_NTD(export, indicator, wkbk, value, group, comp)




In [ ]:

if by_mode:

    # Set params
    indicator = 'Transit_6'
    wkbk = 'Revenue Sources'
    value = 'Funds Expended on Operations Per Capita'
    group = 'Source'
    comp = 'self'

    # Plot
    plot_NTD(export, indicator, wkbk, value, group, comp)




Transit_8 Vehicle Inventory

In [ ]:


if by_mode:

    # Set params
    indicator = 'Transit_8'
    wkbk = 'Vehicle Inventories'
    value = 'Weighted Average Miles Per Capita'
    group = 'Vehicle Type'
    comp = 'peers_sacrt'

    # Plot
    plot_NTD(export, indicator, wkbk, value, group, comp)




In [ ]:

if by_mode:

    # Set params
    indicator = 'Transit_8'
    wkbk = 'Vehicle Inventories'
    value = 'Weighted Average Miles Per Capita'
    group = 'Vehicle Type'
    comp = 'self'

    # Plot
    plot_NTD(export, indicator, wkbk, value, group, comp)


